In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [ ]:
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import json
from pathlib import Path
import einops
from einops import rearrange
from sklearn.decomposition import PCA
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from source.models.classification.my_knet import MyAKOrN, AKOrNResNet
from source.models.classification.analysis_utils import(
    AKOrNDynamicalAnalyzer,
    AKOrNStaticAnalyzer,
    reshape_blocks_to_tensor,
    reshape_tensor_to_blocks
)
from source.data.augs import augmentation_strong

# Set style for better plots

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    #project_root = Path.cwd()
    project_root = Path.cwd().parent
elif device.type == 'cuda':
    project_root = Path.cwd()
print(f"Using device: {device}")
print(f"Project root: {project_root}")

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


In [ ]:
inch = 128
outch = 128
ro_N = 2

invconv = nn.Conv2d(
        inch,
        outch * ro_N,
        kernel_size=3,
        stride=1,
        padding=1,
    )
x_T = invconv(torch.randn(64, 128, 32, 32))
print(x_T.shape)
invconv_x_T = torch.randn(64, 128*2, 32, 32)
invconv_x_T.unflatten(1, (128,-1)).shape

In [ ]:
ch_in = 128
ch_out = 128
n = 2
ksize = 9

# J_00 = nn.Parameter(torch.randn(ch_out//n, ch_in//n, ksize, ksize) * 0.01)
# J_01 = nn.Parameter(torch.randn(ch_out//n, ch_in//n, ksize, ksize) * 0.01)
J_00 = nn.Parameter(torch.randn(ch_out//n * ch_in//n * ksize * ksize, ) * 0.01)
J_01 = nn.Parameter(torch.randn(ch_out//n * ch_in//n * ksize * ksize, ) * 0.01)

In [ ]:
J_00.shape

In [ ]:
torch.stack([J_00, J_01], dim=1).shape

In [ ]:
J_tensor = torch.stack([
            torch.stack([J_00, J_01], dim=1),  # J_{00}, J_{01}
            torch.stack([-J_01, J_00], dim=1) # J_{10}, J_{11}
        ], dim=2)

In [ ]:
J_tensor.shape

In [ ]:
J_tensor.shape

In [ ]:
torch.stack([J_00, J_01], dim=1).shape

In [ ]:
# Load the best model checkpoint
results_dir = Path("results")
checkpoint_path = project_root / results_dir / "20250704_570979.opbs/my_akorn_cifar10_final.pth"
config_path = project_root / results_dir / "20250704_570979.opbs/parameters.json"

# Load configuration
with open(config_path, 'r') as f:
    config = json.load(f)

print("Model Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
if 'epoch' in str(checkpoint_path):
    print(f"\nLoaded checkpoint from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
elif 'final' in str(checkpoint_path):
    print(f"\nLoaded final checkpoint with accuracy {checkpoint['final_accuracy']:.2f}%")

# Create model with same configuration
model =MyAKOrN(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=config['T'],
    J=config['J'],
    J_bias=config['J_bias'],
    ksizes=config['ksizes'],
    ro_ksize=config['ro_ksize'],
    ro_N=config['ro_N'],
    norm=config['norm'],
    c_norm=config['c_norm'],
    gamma=config['gamma'],
    use_omega=config['use_omega'],
    init_omg=config['init_omg'],
    global_omg=config['global_omg'],
    learn_omg=config['learn_omg'],
    ensemble=config['ensemble']
).to(device)

# Load state dict
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\nModel loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
Layer0Analysis = AKOrNStaticAnalyzer(model, 0)
J_blocks = Layer0Analysis.extract_connectivity_blocks()
J_weights = Layer0Analysis.extract_connectivity_weights()
J_tensor = J_weights['weight']

#J_blocks = torch.from_numpy(np.array(J_blocks))

In [ ]:
print("J_blocks shape:", J_blocks.shape)
print("J_tensor shape:", J_tensor.shape)

In [ ]:
J_blocks_check = reshape_tensor_to_blocks(128, 128, 9, 9, 2, J_tensor)

In [ ]:
np.array_equal(J_blocks_check, J_blocks)

In [ ]:
J_tensor_check = reshape_blocks_to_tensor(J_blocks, 2, 128, 128, 9, 9) 

In [ ]:
np.array_equal(J_tensor_check, J_tensor)

In [ ]:
J_blocks.reshape(128//2, 128//2, 9, 9, 2, 2).transpose(0, 4, 1, 5, 2, 3).shape

In [ ]:
J_weights.keys()

In [ ]:
J_weights["weight"].shape